# vidaudit — Qwen2.5-VL smoke test

End-to-end smoke run of the **canonical eval backend** ([Qwen/Qwen2.5-VL-3B-Instruct](https://huggingface.co/Qwen/Qwen2.5-VL-3B-Instruct)) on Colab.

The point of this notebook is to verify that, on a fresh Colab T4, the full pipeline works without paid APIs:

1. Install vidaudit with the `qwen` extra (transformers + torch + accelerate).
2. Download a short test video and a hand-written descriptions file.
3. Run `vidaudit audit --backend qwen` and view the report.

**Runtime:** Set the runtime to T4 GPU (Runtime → Change runtime type → T4 GPU). The 3B checkpoint is ~7 GB in fp16, which fits comfortably. If you only have a smaller GPU, use the optional 4-bit cell at the bottom.

**Reproducibility:** for a canonical eval, pin a commit SHA via `--qwen-revision <sha>`. This notebook leaves it unpinned for ease of running.

## 1. Install vidaudit + Qwen extras

Clone the repo and install in editable mode. The `[qwen]` extra pulls in `transformers>=4.49`, `torch`, and `accelerate`.

In [2]:
!git clone https://github.com/shaneopatrick/vidaudit.git /content/vidaudit
%cd /content/vidaudit
!pip install -q -e '.[qwen]'
!python -m spacy download en_core_web_sm

Cloning into '/content/vidaudit'...
remote: Enumerating objects: 108, done.
remote: Counting objects: 100% (108/108), done.
remote: Compressing objects: 100% (55/55), done.
remote: Total 108 (delta 45), reused 103 (delta 45), pack-reused 0 (from 0)
Receiving objects: 100% (108/108), 347.43 KiB | 19.30 MiB/s, done.
Resolving deltas: 100% (45/45), done.
/content/vidaudit
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Installing backend dependencies ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for vidaudit (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 33.7 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the pack

In [3]:
import torch

print("torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

torch: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4


## 2. Grab a sample video + descriptions

We use a short Creative-Commons clip and write a small hand-crafted descriptions file with one accurate claim and one obvious hallucination so the audit has something to flag.

In [8]:
import json
from pathlib import Path

DATA = Path("/content/sample_data")
DATA.mkdir(exist_ok=True)

video_url = "https://test-videos.co.uk/vids/bigbuckbunny/mp4/h264/720/Big_Buck_Bunny_720_10s_1MB.mp4"
!wget -q -O {DATA}/clip.mp4 {video_url}
!ffprobe -v error -show_entries format=duration -of default=nw=1:nk=1 {DATA}/clip.mp4

descs = [
    {
        "timestamp_start": 2.0,
        "timestamp_end": 4.0,
        "description": "A large rabbit yawns near a tree in a green meadow.",
    },
    {
        "timestamp_start": 6.0,
        "timestamp_end": 8.0,
        "description": "A red sports car drives along a highway past the Eiffel Tower.",
    },
]
(DATA / "descs.json").write_text(json.dumps(descs, indent=2))
print("Wrote", DATA / "descs.json")

10.000000
Wrote /content/sample_data/descs.json


In [9]:
  !ls -la /content/sample_data/clip.mp4
  !file /content/sample_data/clip.mp4
  !head -c 200 /content/sample_data/clip.mp4

-rw-r--r-- 1 root root 969201 Mar 22  2019 /content/sample_data/clip.mp4
/content/sample_data/clip.mp4: ISO Media, MP4 Base Media v1 [ISO 14496-12:2003]
ttrak   \tkhd                    '            

## 3. Run the audit

First time through, this downloads the ~7 GB Qwen2.5-VL-3B-Instruct checkpoint into the Colab disk. Subsequent runs are fast — frames and verification verdicts are both cached on disk.

In [12]:
!vidaudit audit \
    --video /content/sample_data/clip.mp4 \
    --descriptions /content/sample_data/descs.json \
    --output /content/report.json \
    --backend qwen \
    --verbose

INFO numexpr.utils — NumExpr defaulting to 2 threads.
INFO vidaudit.vlm.qwen_vl — Loading Qwen2.5-VL model Qwen/Qwen2.5-VL-3B-Instruct (revision=None)…
INFO httpx — HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-VL-3B-Instruct/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
INFO httpx — HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2.5-VL-3B-Instruct/66285546d2b821cf421d4f5eb2576359d3770cd3/config.json "HTTP/1.1 200 OK"
INFO httpx — HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-VL-3B-Instruct/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
INFO httpx — HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2.5-VL-3B-Instruct/66285546d2b821cf421d4f5eb2576359d3770cd3/config.json "HTTP/1.1 200 OK"
INFO httpx — HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-VL-3B-Instruct/resolve/main/model.safetensors "HTTP/1.1 404 Not Found"
INFO httpx — HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5

In [13]:
import json

with open("/content/report.json") as f:
    report = json.load(f)

print(f"Backend:    {report['metadata']['backend']}")
print(f"Grounding:  {report['summary']['overall_grounding_score']:.2f}")
print(f"Flagged:    {report['summary']['descriptions_flagged']}/{report['summary']['total_descriptions']}")

for seg in report["segments"]:
    print(f"\n[{seg['timestamp_start']:.1f}-{seg['timestamp_end']:.1f}] {seg['verdict']}: {seg['description']}")
    # Each claim is a ClaimResult: {claim: {text, claim_type, ...},
    # verification: {verdict, confidence, evidence}, flagged}.
    for c in seg["claims"]:
        v = c["verification"]
        flag = " FLAG" if c["flagged"] else ""
        print(f"  - ({v['verdict']}, conf={v['confidence']:.2f}){flag} {c['claim']['text']}")

Backend:    Qwen/Qwen2.5-VL-3B-Instruct
Grounding:  0.60
Flagged:    1/2

[2.0-4.0] clean: A large rabbit yawns near a tree in a green meadow.
  - (supported, conf=0.90) large rabbit yawns
  - (supported, conf=0.90) tree
  - (supported, conf=0.90) green meadow

[6.0-8.0] full_hallucination: A red sports car drives along a highway past the Eiffel Tower.
  - (unsupported, conf=0.95) FLAG highway
  - (unsupported, conf=0.98) FLAG the Eiffel Tower


## 4. (Optional) 4-bit path for smaller GPUs

If you're not on a T4, run the cell below to install `bitsandbytes` and re-audit with `--qwen-4bit`. VRAM drops from ~7 GB to ~4 GB; accuracy delta on this verifier task is small but non-zero (a worth-comparing data point — see BACKLOG).

In [ ]:
# !pip install -q bitsandbytes
# !vidaudit audit \
#     --video /content/sample_data/clip.mp4 \
#     --descriptions /content/sample_data/descs.json \
#     --output /content/report_4bit.json \
#     --backend qwen \
#     --qwen-4bit \
#     --verbose